# Zero-Shot Text Restoration via Masked Diffusion Models -- Ablation Study

Colab entrypoint (Phase 4). Runs the full ablation study over masking ratios, corruption topologies, and models (Qwen2.5-Coder FIM vs. MDLM), with real models/data (`use_mock=False`). Do not run this notebook's cells locally -- they download and load multi-GB model checkpoints and the full WikiText-103 dataset.

## 1. Colab setup

In [ ]:
!git clone https://github.com/TommasoFederici/Zero-Shot-text-Restoration-via-Masked-Diffusion-Model.git
%cd Zero-Shot-text-Restoration-via-Masked-Diffusion-Model
!pip install -r requirements.txt

## 2. Imports

In [ ]:
import itertools
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import RandomTokenMasking, SpanMasking, load_dataset_texts
from src.eval import evaluate_batch
from src.models import build_restorer

## 3. Ablation grid configuration

`use_mock=False` is used everywhere below -- this is the only place in the project where real models/data should be loaded, per `CLAUDE.md`'s local-testing rule. Do not copy `use_mock=False` calls back into `src/` scripts or tests.

In [ ]:
MASKING_RATIOS = [0.1, 0.3, 0.5]
CORRUPTION_TYPES = ["random_token", "span"]
MDLM_TIMESTEPS = [4, 8, 16]

# Start small for an end-to-end dry run, then scale up (e.g. 50-100) for the final run.
NUM_TEXTS = 20
SEED = 42

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CORRUPTORS = {
    "random_token": lambda: RandomTokenMasking(seed=SEED),
    "span": lambda: SpanMasking(seed=SEED),
}

## 4. Data loading (real WikiText-103)

In [ ]:
texts = load_dataset_texts(
    use_toy=False, split="train", max_samples=NUM_TEXTS, min_words=10
)
print(f"Loaded {len(texts)} texts")

## 5. Sanity check

Run one real restoration per model on a single text before launching the full ablation loop. This is the first place to catch MDLM mask-token-id resolution issues or Qwen loading/OOM problems, without burning the full grid's runtime.

For MDLM specifically: inspect `tokenizer.mask_token` / `model.config` here if `resolve_mask_token_id` raises, since the absorbing/mask token id used by `kuleshov-group/mdlm-owt` was not verified ahead of time and may need adjustment in `src/mdlm_utils.py`.

In [ ]:
sanity_corruption = SpanMasking(seed=SEED).corrupt(texts[0], 0.3)

qwen_sanity = build_restorer("qwen_fim", use_mock=False).restore(sanity_corruption)
print("Qwen restored text:", qwen_sanity.restored_text)

mdlm_sanity = build_restorer("mdlm", use_mock=False, num_timesteps=8).restore(
    sanity_corruption
)
print("MDLM restored text:", mdlm_sanity.restored_text)

## 6. Main ablation loop

Loops over masking ratio x corruption type x model type, with an inner loop over `MDLM_TIMESTEPS` only for MDLM (Qwen has no timesteps axis). This is by far the longest-running cell -- progress is printed per combination.

If you hit GPU OOM running both models in the same session, restructure this loop to be model-outer and call `del model; torch.cuda.empty_cache()` between the Qwen and MDLM phases, or switch `QwenFIMRestorer`'s `model_name` to a smaller variant (e.g. `Qwen/Qwen2.5-Coder-1.5B`).

In [ ]:
all_results = []

for ratio, corruption_type in itertools.product(MASKING_RATIOS, CORRUPTION_TYPES):
    corruptor = CORRUPTORS[corruption_type]()
    corruptions = [corruptor.corrupt(text, ratio) for text in texts]

    for model_type in ["qwen_fim", "mdlm"]:
        timestep_values = MDLM_TIMESTEPS if model_type == "mdlm" else [None]

        for num_timesteps in timestep_values:
            start = time.perf_counter()

            kwargs = {"num_timesteps": num_timesteps} if num_timesteps is not None else {}
            restorer = build_restorer(model_type, use_mock=False, **kwargs)
            restorations = [restorer.restore(c) for c in corruptions]
            evaluations = evaluate_batch(restorations, use_mock=False)
            all_results.extend(evaluations)

            elapsed = time.perf_counter() - start
            print(
                f"{model_type} | {corruption_type} | ratio={ratio} "
                f"| steps={num_timesteps} | n={len(texts)} | {elapsed:.1f}s"
            )

## 7. Aggregate results into a DataFrame

In [ ]:
df = pd.DataFrame(
    [
        {
            "model_name": r.model_name,
            "corruption_type": r.corruption_type,
            "masking_ratio_requested": r.masking_ratio_requested,
            "masking_ratio_actual": r.masking_ratio_actual,
            "num_timesteps": r.inference_config.get("num_timesteps"),
            "rouge_l_f1": r.rouge_l_f1,
            "bertscore_f1": r.bertscore_f1,
            "bertscore_precision": r.bertscore_precision,
            "bertscore_recall": r.bertscore_recall,
            "perplexity": r.perplexity,
            "latency_seconds": r.restoration.latency_seconds,
        }
        for r in all_results
    ]
)
df.head()

## 8. Export results

In [ ]:
df.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
df.to_json(OUTPUT_DIR / "ablation_results.json", orient="records", indent=2)

## 9. Plots

In [ ]:
sns.set_theme(style="whitegrid")

# Quality vs. masking ratio, one line per (model, corruption_type)
for metric in ["rouge_l_f1", "bertscore_f1"]:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.lineplot(
        data=df,
        x="masking_ratio_requested",
        y=metric,
        hue="model_name",
        style="corruption_type",
        marker="o",
        ax=ax,
    )
    ax.set_title(f"{metric} vs. masking ratio")
    fig.savefig(OUTPUT_DIR / f"{metric}_vs_ratio.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Corruption topology comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["rouge_l_f1", "bertscore_f1"]):
    sns.barplot(data=df, x="corruption_type", y=metric, hue="model_name", ax=ax)
    ax.set_title(f"{metric} by corruption type")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "corruption_type_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# MDLM quality and latency vs. num_timesteps (inference-efficiency tradeoff)
mdlm_df = df[df["model_name"].str.contains("mdlm", case=False)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.lineplot(
    data=mdlm_df,
    x="num_timesteps",
    y="bertscore_f1",
    hue="masking_ratio_requested",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("MDLM bertscore_f1 vs. num_timesteps")

sns.lineplot(
    data=mdlm_df,
    x="num_timesteps",
    y="latency_seconds",
    hue="masking_ratio_requested",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("MDLM latency vs. num_timesteps")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mdlm_timesteps_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Qwen vs. MDLM headline summary, averaged across the whole grid
summary = df.groupby("model_name")[["rouge_l_f1", "bertscore_f1", "perplexity"]].mean()
print(summary)

fig, ax = plt.subplots(figsize=(6, 4))
summary[["rouge_l_f1", "bertscore_f1"]].plot(kind="bar", ax=ax)
ax.set_title("Qwen vs. MDLM -- average quality across the ablation grid")
fig.savefig(OUTPUT_DIR / "model_headline_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Optional: perplexity distribution by model (unbounded, lower-is-better --
# kept on its own axes rather than alongside ROUGE-L/BERTScore)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="model_name", y="perplexity", ax=ax)
ax.set_title("Perplexity distribution by model")
fig.savefig(OUTPUT_DIR / "perplexity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()